In [ ]:
# Enterprise FinTech Payment Intelligence Platform
## Phase 4 - Deployment Preparation
### Notebook 07 - Model Deployment Preparation

**Objective:**
Package the winning champion model (Random Forest), preprocessing scaler, feature list, and metadata into a final deployment-ready format. Demonstrate a single-record inference prediction.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

In [2]:
print("Loading Random Forest (Champion Model)...")
champion_model = joblib.load("RandomForest_Model.pkl")

print("Saving as Deployment Artifact...")
joblib.dump(champion_model, "Champion_Fraud_Model.pkl")
print("Champion Model Saved: Champion_Fraud_Model.pkl")

Loading Random Forest (Champion Model)...
Saving as Deployment Artifact...
Champion Model Saved: Champion_Fraud_Model.pkl


In [3]:
print("Generating Scaler Artifact...")
# Load the raw engineered data to fit a fresh deployment scaler
df = pd.read_csv("feature_engineered_dataset.csv")

# Drop the columns we excluded during training
X = df.drop(columns=["TransactionID", "IsFraud", "SourceAccountID", "DestinationAccountID"])

# Fit the scaler and save it
scaler = StandardScaler()
scaler.fit(X)

joblib.dump(scaler, "Scaler.pkl")
print("Scaler Saved: Scaler.pkl")

Generating Scaler Artifact...
Scaler Saved: Scaler.pkl


In [4]:
print("Saving Feature List...")
feature_list = list(X.columns)

joblib.dump(feature_list, "Feature_List.pkl")
print(f"Feature List Saved ({len(feature_list)} features): Feature_List.pkl")

Saving Feature List...
Feature List Saved (20 features): Feature_List.pkl


In [5]:
print("Generating Model Metadata...")

metadata = {
    "project_name": "Enterprise FinTech Payment Intelligence Platform",
    "model_type": "Random Forest Classifier",
    "version": "1.0.0",
    "author": "Anto Abraham",
    "input_features": len(feature_list),
    "primary_metrics": {
        "Precision": 0.9572,
        "Recall": 0.9945,
        "F1_Score": 0.9755,
        "ROC_AUC": 0.9988
    },
    "description": "Champion model for detecting fraudulent FinTech transactions."
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
    
print("Metadata Saved: model_metadata.json")

Generating Model Metadata...
Metadata Saved: model_metadata.json


In [6]:
print("="*60)
print("🚀 SIMULATING REAL-TIME INFERENCE API")
print("="*60)

# 1. Load Deployment Artifacts
deploy_model = joblib.load("Champion_Fraud_Model.pkl")
deploy_scaler = joblib.load("Scaler.pkl")
deploy_features = joblib.load("Feature_List.pkl")

# 2. Simulate an incoming transaction from a mobile app/API
incoming_transaction = {
    'DayNumber': 14,
    'HourOfSimulation': 21,
    'Amount': 850000.00,
    'OldBalanceOrig': 850000.00,
    'NewBalanceOrig': 0.00,
    'OldBalanceDest': 1500.00,
    'NewBalanceDest': 851500.00,
    'IsFlaggedFraud': 0,
    'OriginBalanceChange': 850000.00,
    'DestinationBalanceChange': 850000.00,
    'AmountToOriginBalanceRatio': 1.0, # 100% of balance being transferred
    'HighValueTransaction': 1,
    'BalanceGap': 848500.00,
    'PeriodOfDay_Evening': 1,
    'PeriodOfDay_Morning': 0,
    'PeriodOfDay_Night': 0,
    'TransactionType_CASH_OUT': 0,
    'TransactionType_DEBIT': 0,
    'TransactionType_PAYMENT': 0,
    'TransactionType_TRANSFER': 1 # High risk transfer
}

# 3. Format for prediction
input_df = pd.DataFrame([incoming_transaction])[deploy_features]
scaled_input = deploy_scaler.transform(input_df)

# 4. Predict
prediction = deploy_model.predict(scaled_input)[0]
probability = deploy_model.predict_proba(scaled_input)[0][1]

print(f"Transaction Amount : ${incoming_transaction['Amount']:,.2f}")
print(f"Fraud Probability  : {probability * 100:.2f}%")
print(f"Final Decision     : {'🚨 FRAUD DETECTED - BLOCK TRANSACTION' if prediction == 1 else '✅ APPROVED'}")

🚀 SIMULATING REAL-TIME INFERENCE API
Transaction Amount : $850,000.00
Fraud Probability  : 36.00%
Final Decision     : ✅ APPROVED
